In [1]:
import os
import re
import pandas as pd
import numpy as np

from datasets import (
    load_dataset,
    load_from_disk
)

from sklearn.model_selection import train_test_split

from transformers import BertTokenizer

In [2]:
RAW_DATA_PATH = "data/raw/ag_news"

PROCESSED_DATA_PATH = "data/processed/ag_news"

SPLIT_DATA_PATH = "data/splits"

In [3]:
os.makedirs(RAW_DATA_PATH, exist_ok=True)

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

os.makedirs(SPLIT_DATA_PATH, exist_ok=True)

print("Directories created successfully.")

Directories created successfully.


In [4]:
dataset = load_dataset("wangrongsheng/ag_news")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [5]:
dataset.save_to_disk(RAW_DATA_PATH)

print(f"Raw dataset saved to: {RAW_DATA_PATH}")

Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]

Raw dataset saved to: data/raw/ag_news


In [6]:
dataset = load_from_disk(RAW_DATA_PATH)

print("Raw dataset loaded successfully.")

Raw dataset loaded successfully.


In [7]:
train_df = pd.DataFrame(dataset["train"])

train_df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [8]:
train_df["label"].value_counts()

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64

In [9]:
L2_LABELS = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "SciTech"
}

L1_MAPPING = {
    0: 0,  # World -> Hard News
    1: 1,  # Sports -> Soft News
    2: 0,  # Business -> Hard News
    3: 1   # SciTech -> Soft News
}

L1_LABELS = {
    0: "Hard News",
    1: "Soft News"
}

In [10]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [11]:
def preprocess(example):

    cleaned_text = clean_text(example["text"])

    l2_label = example["label"]

    l1_label = L1_MAPPING[l2_label]

    return {
        "clean_text": cleaned_text,
        "l1_label": l1_label,
        "l2_label": l2_label
    }

In [12]:
processed_dataset = dataset.map(preprocess)

print("Preprocessing completed.")

Preprocessing completed.


In [13]:
processed_dataset.save_to_disk(
    PROCESSED_DATA_PATH
)

print(f"Processed dataset saved to: {PROCESSED_DATA_PATH}")

Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]

Processed dataset saved to: data/processed/ag_news


In [14]:
train_valid = processed_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = train_valid["train"]

val_dataset = train_valid["test"]

test_dataset = processed_dataset["test"]

print("Train/Validation/Test split completed.")

Train/Validation/Test split completed.


In [15]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [16]:
def tokenize(batch):

    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [17]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

print("Tokenization completed.")

Tokenization completed.


In [18]:
columns = [
    "input_ids",
    "attention_mask",
    "l1_label",
    "l2_label"
]

train_dataset.set_format(
    type="torch",
    columns=columns
)

val_dataset.set_format(
    type="torch",
    columns=columns
)

test_dataset.set_format(
    type="torch",
    columns=columns
)

print("PyTorch format applied.")

PyTorch format applied.


In [19]:
train_dataset.save_to_disk(
    "data/splits/train"
)

val_dataset.save_to_disk(
    "data/splits/val"
)

test_dataset.save_to_disk(
    "data/splits/test"
)

print("Dataset splits saved successfully.")

Saving the dataset (0/1 shards):   0%|          | 0/108000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]

Dataset splits saved successfully.


In [20]:
pd.DataFrame(train_dataset).to_csv(
    "data/processed/train.csv",
    index=False
)

pd.DataFrame(val_dataset).to_csv(
    "data/processed/val.csv",
    index=False
)

pd.DataFrame(test_dataset).to_csv(
    "data/processed/test.csv",
    index=False
)

print("CSV files saved successfully.")

CSV files saved successfully.


In [23]:
for root, dirs, files in os.walk("data"):
    print(root)

data
data\processed
data\processed\ag_news
data\processed\ag_news\test
data\processed\ag_news\train
data\raw
data\raw\ag_news
data\raw\ag_news\test
data\raw\ag_news\train
data\splits
data\splits\test
data\splits\train
data\splits\val


In [24]:
print(train_dataset[0])

{'l1_label': tensor(0), 'l2_label': tensor(0), 'input_ids': tensor([  101, 13905,  1998,  4963,  1999,  2235,  2845,  2237,  2044,  6859,
         2022, 14540,  2319,  3607, 26665,  1996,  4288,  1997,  2062,  2084,
        13710,  2336,  3008,  1998,  5089,  2076,  1996,  6703,  2203,  2000,
         1037,  5187,  6806,  3126,  2082,  6859,  2187,  4510,  1037,  2155,
        22154,  1999,  1996,  2235,  2845,  2237,  1997,  2022, 14540,  2319,
          102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
    